In [ ]:
"""
TFIDFV3: TFIDFV2 + leetspeak repair. No training.

Same as V1, except every text is cleaned first (zero-width chars, full-width,
Cyrillic/Greek lookalikes, accents, base64) so obfuscated text maps back onto
the vocabulary the model learns from.

Outputs go to Data/TFIDFV3/ :
    X_main.npz, X_test_notinject.npz, X_test_obfuscated.npz, X_test_temporal.npz
    meta_main.csv, meta_test_*.csv
    feature_names.csv
    tfidf_word.joblib, tfidf_char.joblib   (needed later by the React backend)
"""

import os
import re
import base64
import unicodedata

import joblib
import numpy as np
import pandas as pd
import scipy.sparse as sp
from sklearn.feature_extraction.text import TfidfVectorizer

# ----------------------------------------------------------------------
# Config
# ----------------------------------------------------------------------
DATA_DIR = "../Data"
OUT_DIR = "../Data/TFIDFV3"
TRAIN = "train"                      # value in the `split` column used for fitting
STRESS = ["test_notinject", "test_obfuscated", "test_temporal"]
MAX_WORD = 200_000
MAX_CHAR = 300_000

os.makedirs(OUT_DIR, exist_ok=True)

# ----------------------------------------------------------------------
# Normalizer
# (for the React backend, copy this block into normalize.py)
# ----------------------------------------------------------------------
HOMOGLYPHS = {
    # Cyrillic lowercase
    "а": "a", "в": "b", "с": "c", "ԁ": "d", "е": "e", "һ": "h", "і": "i",
    "ј": "j", "к": "k", "ӏ": "l", "м": "m", "н": "h", "о": "o", "р": "p",
    "ѕ": "s", "т": "t", "υ": "u", "ѵ": "v", "ԝ": "w", "х": "x", "у": "y",
    # Cyrillic uppercase
    "А": "A", "В": "B", "С": "C", "Е": "E", "Н": "H", "І": "I", "Ј": "J",
    "К": "K", "М": "M", "О": "O", "Р": "P", "Ѕ": "S", "Т": "T", "Х": "X",
    "У": "Y", "Ԁ": "D",
    # Greek
    "α": "a", "ο": "o", "ν": "v", "ρ": "p", "τ": "t", "ι": "i", "κ": "k",
    "Α": "A", "Β": "B", "Ε": "E", "Ζ": "Z", "Η": "H", "Ι": "I", "Κ": "K",
    "Μ": "M", "Ν": "N", "Ο": "O", "Ρ": "P", "Τ": "T", "Υ": "Y", "Χ": "X",
}
HOMOGLYPH_TABLE = str.maketrans(HOMOGLYPHS)

INVISIBLE = re.compile(
    "[\u200b\u200c\u200d\u200e\u200f\u2060\u2061\u2062\u2063\u2064\ufeff\u00ad]"
)
B64_RE = re.compile(r"(?<![A-Za-z0-9+/=])[A-Za-z0-9+/]{16,}={0,2}(?![A-Za-z0-9+/=])")


# Leetspeak (1gnor3 -> ignore). Applied ONLY to a text that has several
# mixed letter+digit tokens, so normal text like "mp3" or "4k" is left alone.
LEET = str.maketrans({"0": "o", "1": "i", "3": "e", "4": "a", "5": "s", "7": "t"})
LEET_TOKEN = re.compile(
    r"\b(?=[A-Za-z0-9]{1,15}\b)(?=[A-Za-z0-9]*[A-Za-z])(?=[A-Za-z0-9]*[013457])[A-Za-z0-9]+\b"
)                                # {1,15}: long tokens (base64 blobs) are skipped
MIN_LEET_TOKENS = 3


def _fix_leet(text):
    if len(LEET_TOKEN.findall(text)) < MIN_LEET_TOKENS:
        return text
    return LEET_TOKEN.sub(lambda m: m.group(0).translate(LEET), text)


def _decode_b64_blobs(text):
    """Append decoded text of any base64 blobs (keeps the original too)."""
    parts = []
    for blob in B64_RE.findall(text):
        try:
            raw = base64.b64decode(blob + "=" * (-len(blob) % 4), validate=True).decode("utf-8")
        except Exception:
            continue
        if raw and sum(c.isprintable() or c.isspace() for c in raw) / len(raw) > 0.95:
            parts.append(raw)
    return (text + " " + " ".join(parts)) if parts else text


def normalize(text):
    if not isinstance(text, str):
        return ""
    text = INVISIBLE.sub("", text)                    # 1. zero-width chars
    text = unicodedata.normalize("NFKC", text)        # 2. full-width -> ascii
    text = text.translate(HOMOGLYPH_TABLE)            # 3. Cyrillic/Greek lookalikes
    text = unicodedata.normalize("NFKD", text)        # 4. split accents off letters
    text = "".join(c for c in text if not unicodedata.combining(c))
    text = _decode_b64_blobs(text)                    # 5. base64 -> text
    text = _fix_leet(text)                            # 6. leetspeak (after base64)
    return re.sub(r"\s+", " ", text).strip()


# ----------------------------------------------------------------------
# Load + normalize
# ----------------------------------------------------------------------
def load(name):
    df = pd.read_csv(f"{DATA_DIR}/{name}.csv")
    raw = df["text"].fillna("").astype(str)
    df["text"] = raw.map(normalize)
    df.attrs["leet_rows"] = int((raw.map(lambda t: _fix_leet(t) != t)).sum())
    return df


sets = {"main": load("dataset_main")}
for s in STRESS:
    sets[s] = load(s)

main = sets["main"]
train_text = main.loc[main["split"] == TRAIN, "text"]
print("train rows:", len(train_text))
print(main["split"].value_counts(), "\n")

# ----------------------------------------------------------------------
# Fit vectorizers on TRAIN only
# ----------------------------------------------------------------------
word_vec = TfidfVectorizer(
    analyzer="word", ngram_range=(1, 2), lowercase=True,
    min_df=2, max_df=0.95, sublinear_tf=True, max_features=MAX_WORD,
    token_pattern=r"(?u)\b\w+\b", dtype=np.float32,
)
char_vec = TfidfVectorizer(
    analyzer="char_wb", ngram_range=(3, 5), lowercase=True,
    min_df=3, sublinear_tf=True, max_features=MAX_CHAR, dtype=np.float32,
)
word_vec.fit(train_text)
char_vec.fit(train_text)

feature_names = (
    ["w_" + f for f in word_vec.get_feature_names_out()]
    + ["c_" + f for f in char_vec.get_feature_names_out()]
)
print("total features:", len(feature_names), "\n")

joblib.dump(word_vec, f"{OUT_DIR}/tfidf_word_V3.joblib")
joblib.dump(char_vec, f"{OUT_DIR}/tfidf_char_V3.joblib")
pd.Series(feature_names, name="feature").to_csv(f"{OUT_DIR}/feature_names.csv", index=False)

# ----------------------------------------------------------------------
# Transform + save every dataset
# ----------------------------------------------------------------------
meta_cols = ["uid", "split", "label", "attack_type", "family_id", "source"]

print(f"{'dataset':18s} {'shape':>20s} {'avg non-zeros/row':>18s} {'rows leet-fixed':>16s}")
for name, df in sets.items():
    X = sp.hstack([word_vec.transform(df["text"]),
                   char_vec.transform(df["text"])]).tocsr()
    sp.save_npz(f"{OUT_DIR}/X_{name}.npz", X)
    df[[c for c in meta_cols if c in df.columns]].to_csv(f"{OUT_DIR}/meta_{name}.csv", index=False)
    print(f"{name:18s} {str(X.shape):>20s} {X.getnnz(axis=1).mean():>18.1f} {df.attrs['leet_rows']:>16d}")

print(f"\nDone. Files saved in {OUT_DIR}/")

train rows: 9634
split
train    9634
test     1928
val      1927
Name: count, dtype: int64 

total features: 210888 

dataset                           shape  avg non-zeros/row  rows leet-fixed
main                    (13489, 210888)              738.1              272
test_notinject            (339, 210888)              178.4                0
test_obfuscated          (1800, 210888)              285.8              297
test_temporal             (485, 210888)             3296.8               29

Done. Files saved in Data/TFIDFV3/
